### DQN

#### 简介
- 表格法 -> 函数法 使用函数拟合方法来估计Q值, DQN用来解决连续状态下离散动作的问题
- DQN本质是使用深度神经网络近似动作价值函数，解决Q-learning无法有效处理大规模或连续状态空间的问题
- DQN拉开了深度强化学习的序幕, 是深度强化学习的基础

#### DQN 算法
- 定义：DQN 是一种基于 Q-learning 的深度强化学习算法，通过神经网络近似动作价值函数 $Q(s,a)$, 并利用 TD 目标和梯度下降更新网络参数
- 本质: 降低 $Loss$ -> 更新 $w$ -> 间接更新 $Q$

核心公式: 
- Q-Learning更新公式: 

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \left[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\right]
$$

- DQN的损失函数: 

$$
\omega^* = \arg\min_{\omega} \frac{1}{2N} \sum_{i=1}^{N}
\left[
Q_{\omega}(s_i,a_i) -
\left(r_i + \gamma \max_{a'} Q_{\omega}(s_i',a')\right)
\right]^2
$$





#### 经验回放
- 定义: 将智能体与环境交互产生的经验 $(s,a,r,s',done)$ 存入经验回放池（Replay Buffer）, 训练时随机抽取一个批次（Mini-batch）更新 Q 网络
- 流程: 环境交互 -> 存入Replay Buffer -> 随机采样Mini-batch -> 训练Q网络
- 作用: 降低连续交互样本之间的相关性，让训练数据更接近独立采样; 重复利用历史经验，提高样本利用率



#### 目标网络
- 定义: DQN 使用两个结构相同的 Q 网络，当前网络负责预测 Q 值并通过梯度下降更新参数，目标网络负责计算 TD 目标且暂时保持参数固定，每隔 $C$ 次更新同步一次参数，以缓解训练目标不断变化导致的不稳定问题
- 训练顺序：环境交互 → 经验回放 → 目标网络计算 TD 目标 → 当前网络计算预测 Q 值 → 计算 Loss → 梯度下降更新当前网络 → 定期同步目标网络


In [ ]:
import random
import gym
import numpy as np
import collections
from tqdm import tqdm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import rl_utils

In [ ]:
class ReplayBuffer:
    """经验回放池"""
    def __init__(self, capacity):  # 初始化经验回放池，capacity表示最大容量
        self.buffer = collections.deque(maxlen=capacity)  # 创建固定容量的双端队列，满后自动删除最旧经验

    def add(self, state, action, reward, next_state, done):            # 定义添加经验的方法
        self.buffer.append((state, action, reward, next_state, done))  # 将一条状态转移经验添加到队列末尾

    def sample(self, batch_size):  # 定义随机采样方法，batch_size表示采样数量
        transitions = random.sample(self.buffer, batch_size)  # 从回放池中不放回地随机抽取batch_size条经验
        state, action, reward, next_state, done = zip(*transitions)  # 将每条经验拆分并按字段分别组合
        return np.array(state), action, reward, np.array(next_state), done  # 返回5组批量数据

    def size(self):  # 定义获取回放池当前容量的方法
        return len(self.buffer)  # 返回当前存储的经验数量
    

In [ ]:
class Qnet(torch.nn.Module):
    """只有一层隐藏层的Q网络"""
    def __init__(self, state_dim, hidden_dim, action_dim):  # 初始化Q网络，传入状态、隐藏层和动作维度
        super(Qnet, self).__init__()  # 调用父类nn.Module的初始化方法
        self.fc1 = torch.nn.Linear(state_dim, hidden_dim)   # 定义第一层全连接层，将状态映射到隐藏层
        self.fc2 = torch.nn.Linear(hidden_dim, action_dim)  # 定义第二层全连接层，将隐藏层映射到动作Q值

    def forward(self, x):        # 定义神经网络的前向传播过程
        x = F.relu(self.fc1(x))  # 输入经过第一层线性变换，再通过ReLU激活函数
        return self.fc2(x)       # 经过第二层线性变换，输出所有动作的Q值

In [ ]:
class DQN:
    """DQN算法"""
    def __init__(self, state_dim, hidden_dim, action_dim, learning_rate, gamma,
                 epsilon, target_update, device):        # 初始化DQN智能体及其超参数
        self.action_dim = action_dim                     # 保存动作空间维度
        self.q_net = Qnet(state_dim, hidden_dim,
                          self.action_dim).to(device)    # 创建当前Q网络并放到指定计算设备
        self.target_q_net = Qnet(state_dim, hidden_dim,
                                 self.action_dim).to(device)  # 创建结构相同的目标Q网络
        self.optimizer = torch.optim.Adam(self.q_net.parameters(),
                                          lr=learning_rate)   # 创建Adam优化器，只优化当前Q网络
        self.gamma = gamma                               # 设置折扣因子
        self.epsilon = epsilon                           # 设置ε-Greedy随机探索概率
        self.target_update = target_update               # 设置目标网络的参数同步间隔
        self.count = 0                                   # 初始化训练更新次数计数器
        self.device = device                             # 保存计算设备，如CPU或GPU

    def take_action(self, state):                        # 根据ε-Greedy策略选择动作
        if np.random.random() < self.epsilon:            # 以ε的概率进行随机探索
            action = np.random.randint(self.action_dim)  # 随机选择一个动作
        else:                                            # 以1-ε的概率使用当前Q网络进行决策
            state = torch.tensor([state], dtype=torch.float).to(self.device)  # 将状态转换成Tensor并增加Batch维度
            action = self.q_net(state).argmax().item()   # 选择预测Q值最大的动作并转换为Python数值
        return action                                    # 返回选择的动作编号

    def update(self, transition_dict):                   # 使用经验回放池采样得到的Mini-batch训练DQN
        states = torch.tensor(transition_dict['states'],
                              dtype=torch.float).to(self.device)                  # 将批量状态转换为Tensor
        actions = torch.tensor(transition_dict['actions']).view(-1, 1).to(
            self.device)                                                          # 将批量动作整理成列向量
        rewards = torch.tensor(transition_dict['rewards'],
                               dtype=torch.float).view(-1, 1).to(self.device)     # 将批量奖励转换成列向量
        next_states = torch.tensor(transition_dict['next_states'],
                                   dtype=torch.float).to(self.device)             # 将批量下一状态转换为Tensor
        dones = torch.tensor(transition_dict['dones'],
                             dtype=torch.float).view(-1, 1).to(self.device)       # 将终止标志转换成0/1列向量

        q_values = self.q_net(states).gather(1, actions)                          # 当前网络计算每条经验中实际动作对应的预测Q值
        max_next_q_values = self.target_q_net(next_states).max(1)[0].view(-1, 1)  # 目标网络计算每个下一状态的最大Q值
        q_targets = rewards + self.gamma * max_next_q_values * (1 - dones)        # 计算TD目标，终止状态不考虑未来价值
        dqn_loss = torch.mean(F.mse_loss(q_values, q_targets))                    # 计算预测Q值与TD目标之间的均方误差

        self.optimizer.zero_grad()  # 清空当前Q网络上一次训练累积的梯度
        dqn_loss.backward()         # 反向传播，计算Loss对当前网络参数的梯度
        self.optimizer.step()       # Adam根据梯度更新当前Q网络的参数

        if self.count % self.target_update == 0:  # 判断是否达到目标网络的同步时机
            self.target_q_net.load_state_dict(
                self.q_net.state_dict())          # 将当前Q网络的参数复制给目标网络
        self.count += 1                           # 训练更新次数加1